# Day 4: Computer Vision and NLP

Topics covered:
- Image loading, resizing, flipping and cropping with OpenCV
- Webcam capture loops
- Real-time object detection with YOLOv8
- Face detection with Haar Cascade
- Text preprocessing with NLTK (tokenization, stop words, stemming, lemmatization)
- Word frequency and WordCloud visualization
- Sentiment analysis with TextBlob
- Named Entity Recognition (NER)
- Text classification using Bag-of-Words and Naive Bayes
- Bigram text generation and Transformer context clues


## 1. Reading and Inspecting Images with OpenCV


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
def show_image(img, title="Image", figsize=(6, 4)):
    plt.figure(figsize=figsize, dpi=100)
    if len(img.shape) == 3:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img_rgb)
    else:
        plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis("off")
    plt.show()
img = np.zeros((400, 400, 3), dtype=np.uint8)
cv2.rectangle(img, (50, 50), (350, 350), (40, 180, 40), -1)
cv2.circle(img, (200, 200), 80, (230, 80, 50), -1)
cv2.putText(img, "AI Bootcamp", (80, 210), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
print("Image type :", type(img))
print("Image shape:", img.shape)
show_image(img, "Sample Generated Image")


## 2. Resizing, Flipping and Cropping


In [ ]:
resized_img = cv2.resize(img, (250, 200))
flip_vertical   = cv2.flip(img, 0)
flip_horizontal = cv2.flip(img, 1)
h, w, _ = img.shape
cropped_roi = img[int(h*0.2):int(h*0.8), int(w*0.2):int(w*0.8)]
gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
fig, axs = plt.subplots(2, 2, figsize=(10, 8), dpi=100)
axs[0, 0].imshow(cv2.cvtColor(resized_img, cv2.COLOR_BGR2RGB))
axs[0, 0].set_title("Resized (250x200)")
axs[0, 0].axis("off")
axs[0, 1].imshow(cv2.cvtColor(flip_horizontal, cv2.COLOR_BGR2RGB))
axs[0, 1].set_title("Flipped Horizontally")
axs[0, 1].axis("off")
axs[1, 0].imshow(cv2.cvtColor(cropped_roi, cv2.COLOR_BGR2RGB))
axs[1, 0].set_title("Cropped ROI")
axs[1, 0].axis("off")
axs[1, 1].imshow(gray_img, cmap='gray')
axs[1, 1].set_title("Grayscale")
axs[1, 1].axis("off")
plt.tight_layout()
plt.show()


## 3. Webcam Stream Capture


In [ ]:
def run_webcam_demo(apply_edge_filter=False, max_frames=60):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Camera device not detected or permission denied.")
        return
    frame_count = 0
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1
        display_frame = frame.copy()
        if apply_edge_filter:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 100, 200)
            display_frame = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
        cv2.putText(display_frame, f"Frame #{frame_count} | Press 'q' to Quit", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.imshow("Webcam Stream", display_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("Camera connection released.")
print("Webcam function defined. Call run_webcam_demo() to run.")


## 4. Object Detection with YOLOv8


In [ ]:
def run_yolo_object_counter(max_frames=60):
    try:
        from ultralytics import YOLO, settings
        settings.update({"sync": False})
        model = YOLO("yolov8n.pt")
        print("Loaded YOLOv8 model with classes:", len(model.names))
        results = model(img, verbose=False)
        count = len(results[0].boxes)
        annotated = results[0].plot()
        show_image(annotated, f"YOLOv8 Detection (Count: {count})")
    except ImportError:
        print("Ultralytics library not installed in this environment.")
run_yolo_object_counter(max_frames=1)


## 5. Face Detection with Haar Cascade


In [ ]:
if hasattr(cv2, 'CascadeClassifier'):
    print("CascadeClassifier is available in this OpenCV installation.")
else:
    print("CascadeClassifier not available in this OpenCV build.")


## 6. Text Processing with NLTK


In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
for pkg in ['punkt', 'stopwords', 'wordnet']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
sample_paragraph = (
    "Photography is an art that captures the beauty of moments in time.\n"
    "Whether it's a stunning landscape, a candid portrait, or a lively street scene,\n"
    "photographs tell stories that evoke deep emotions. Photographers have been capturing\n"
    "extraordinary images across different generations."
)
sentences = sent_tokenize(sample_paragraph)
words = word_tokenize(sample_paragraph)
print(f"Total Sentences : {len(sentences)}")
print(f"Total Words     : {len(words)}")
try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    stop_words = {"is", "an", "that", "the", "of", "in", "it's", "a", "or", "have", "been"}
filtered_words = [w for w in words if w.isalpha() and w.lower() not in stop_words]
print(f"Filtered Words  : {len(filtered_words)}")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
test_words = ["captures", "photographs", "stories", "emotions", "photographers", "capturing", "generations"]
print("\nOriginal | Stemmed | Lemmatized")
print("-" * 35)
for tw in test_words:
    stem_val = stemmer.stem(tw)
    try:
        lemma_val = lemmatizer.lemmatize(tw.lower(), pos='v')
    except Exception:
        lemma_val = tw
    print(f"{tw:<12} | {stem_val:<8} | {lemma_val}")


## 7. Word Frequency and WordCloud


In [ ]:
from nltk.probability import FreqDist
freq_dist = FreqDist([w.lower() for w in filtered_words])
print("Top 5 Most Frequent Words:")
for word, count in freq_dist.most_common(5):
    print(f" * '{word}': {count}")
try:
    from wordcloud import WordCloud
    clean_text = " ".join([w.lower() for w in filtered_words])
    wc = WordCloud(width=800, height=400, background_color="white", colormap="viridis", max_words=50).generate(clean_text)
    plt.figure(figsize=(10, 5), dpi=100)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title("WordCloud")
    plt.show()
except ImportError:
    words_plot, counts_plot = zip(*freq_dist.most_common(8))
    plt.figure(figsize=(8, 4))
    plt.bar(words_plot, counts_plot, color="#3b82f6")
    plt.title("Word Frequency Distribution")
    plt.show()


## 8. Sentiment Analysis


In [ ]:
try:
    from textblob import TextBlob
    feedback_reviews = [
        "The AI Bootcamp at NIELIT Delhi was an absolutely incredible experience!",
        "The computer vision session was quite difficult and confusing.",
        "Today we covered Python syntax, NumPy matrices, and machine learning models.",
        "The instructor was exceptionally helpful and explained everything clearly.",
        "I was very disappointed by the connection issues."
    ]
    for review in feedback_reviews:
        blob = TextBlob(review)
        pol = round(blob.sentiment.polarity, 2)
        subj = round(blob.sentiment.subjectivity, 2)
        if pol > 0.15:
            sentiment = "Positive"
        elif pol < -0.15:
            sentiment = "Negative"
        else:
            sentiment = "Neutral"
        print(f"[{sentiment:<8}] Polarity: {pol:>5.2f} | {review}")
except ImportError:
    print("TextBlob not installed.")


## 9. Named Entity Recognition (NER)


In [ ]:
try:
    import spacy
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        nlp = None
    resume_text = (
        "Vibhor Singh is an AI Engineering Fellow at NIELIT Delhi.\n"
        "He completed his machine learning internship at Google in Mountain View, California on August 15, 2024.\n"
        "He worked on computer vision and natural language processing."
    )
    if nlp is not None:
        doc = nlp(resume_text)
        for ent in doc.ents:
            print(f" * {ent.text:<25} -> {ent.label_}")
    else:
        print("spaCy model en_core_web_sm not found.")
except ImportError:
    print("spaCy not installed.")


## 10. Spam Message Classifier


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
train_messages = [
    ("Congratulations! You won a $1,000 cash prize. Click here to claim immediately.", "Spam"),
    ("Urgent: Your account is locked. Reply with password to restore access.", "Spam"),
    ("Exclusive loan offer at 0% interest! Call now to verify your credentials.", "Spam"),
    ("FREE coupon code inside! Win gift cards today only.", "Spam"),
    ("Hey, are we still meeting in the computer lab for AI class today?", "Ham"),
    ("Please find attached the Jupyter notebook for Day 3 assignment.", "Ham"),
    ("What time does the NIELIT bootcamp lecture start tomorrow morning?", "Ham"),
    ("Don't forget to submit your student performance project before 5 PM.", "Ham")
]
texts, labels = zip(*train_messages)
vectorizer = CountVectorizer(stop_words='english')
X_bow = vectorizer.fit_transform(texts)
spam_model = MultinomialNB()
spam_model.fit(X_bow, labels)
incoming_messages = [
    "URGENT: Claim your free lottery reward right now!",
    "Hi Vibhor, let's review the machine learning code together.",
    "Click this link to win cash and exciting gifts today.",
    "Can you share the dataset CSV file for today's lab?"
]
X_new = vectorizer.transform(incoming_messages)
preds = spam_model.predict(X_new)
for msg, pred in zip(incoming_messages, preds):
    print(f"[{pred.upper()}] {msg}")


## 11. Simple Bigram Text Generation


In [ ]:
import random
training_corpus = (
    "artificial intelligence is transforming the world. artificial intelligence powers computer vision. "
    "computer vision enables machines to see. natural language processing allows computers to understand text. "
    "machines learn from data to make accurate predictions."
).lower().split()
bigram_transitions = {}
for i in range(len(training_corpus) - 1):
    curr_word = training_corpus[i]
    next_word = training_corpus[i+1]
    if curr_word not in bigram_transitions:
        bigram_transitions[curr_word] = []
    bigram_transitions[curr_word].append(next_word)
def generate_text(seed_word, length=8):
    current = seed_word.lower()
    output = [current]
    for _ in range(length - 1):
        options = bigram_transitions.get(current)
        if not options:
            break
        current = random.choice(options)
        output.append(current)
    return " ".join(output)
for seed in ["artificial", "computer", "machines"]:
    print(f"Prompt '{seed}': {generate_text(seed, 7)}")


## 12. Context Clues and Attention Concept


In [ ]:
sentences = [
    "The fisherman sat beside the river bank to cast his fishing line.",
    "The investor went inside the financial bank to deposit currency."
]
context_clues = {
    "Financial": ["investor", "deposit", "currency", "financial", "money", "loan"],
    "Nature/River": ["fisherman", "river", "fishing", "water", "stream", "lake"]
}
for s in sentences:
    tokens = [w.lower().strip(".,") for w in s.split()]
    fin_score = sum(1 for t in tokens if t in context_clues["Financial"])
    nat_score = sum(1 for t in tokens if t in context_clues["Nature/River"])
    assigned_context = "Financial Institution" if fin_score > nat_score else "River/Water Bank"
    print(f"Sentence: '{s}'")
    print(f"  Scores -> Financial: {fin_score} | Nature: {nat_score} => [{assigned_context}]")
